# Лабораторна робота №3: Робота з нейронними мережами. Навчання мережі
**Дисципліна:** Системи штучного інтелекту  
**Тема дослідження:** Динаміка цін на нерухомість в післявоєнній Україні  
**Студент:** Чисельський Владислав Леонідович, група КН-2327Б  
**Інструменти:** Python 3, TensorFlow / Keras, NumPy, Pandas, Matplotlib, Google Colaboratory

## 1. Підключення бібліотек для роботи з нейронною мережею

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Версія TensorFlow:', tf.__version__)
print('Бібліотеки підключено успішно!')

## 2. Завантаження та попередня обробка даних
Формуємо набір даних щодо повоєнного ринку нерухомості з урахуванням 9 ключових факторів:
- **RegionSafety**: індекс безпеки та віддаленості від зони бойових дій (0.3 - 1.0)
- **InfraRecovery**: рівень відновлення критичної та транспортної інфраструктури (0.1 - 1.0)
- **ShelterDistance**: відстань до обладнаного укриття чи підземного паркінгу (м)
- **EnergyAutonomy**: рівень енергонезалежності будинку (0.1 - 1.0)
- **AreaSqm**: площа житла (м²)
- **Rooms**: кількість кімнат (1 - 4)
- **NewBuilding**: стандарт відбудови (1 — сучасна відбудова, 0 — старий фонд)
- **EconomicActivity**: індекс ділової активності та зайнятості міста (0.2 - 1.0)
- **RecoveryQuarter**: квартал повоєнного відновлення (1 - 16)
- **TotalPriceKUSD** (Target): прогнозована вартість житла у тисячах доларів

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print('Навчальних записів:', len(train_df))
print('Тестових записів:  ', len(test_df))
train_df.head()

In [ ]:
X_train_raw = train_df.drop(columns=['TotalPriceKUSD']).values
y_train = train_df['TotalPriceKUSD'].values

X_test_raw = test_df.drop(columns=['TotalPriceKUSD']).values
y_test = test_df['TotalPriceKUSD'].values

mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
std[std == 0] = 1.0

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

print('Нормалізацію виконано!')
print('Розмірність навчального масиву:', X_train.shape)

## 3. Побудова архітектури нейронної мережі TensorFlow Keras

In [ ]:
def create_model():
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],), name='hidden_1'),
        layers.Dense(32, activation='relu', name='hidden_2'),
        layers.Dense(16, activation='relu', name='hidden_3'),
        layers.Dense(1, activation='linear', name='price_output')
    ], name='PostWar_Housing_MLP')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss='mse',
        metrics=['mae']
    )
    return model

model = create_model()
model.summary()

## 4. Експеримент 1: Навчання протягом 100 епох

In [ ]:
model_100 = create_model()
history_100 = model_100.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

pred_100 = model_100.predict(X_test).flatten()
mae_100 = np.mean(np.abs(y_test - pred_100))
rmse_100 = np.sqrt(np.mean((y_test - pred_100)**2))
r2_100 = 1.0 - (np.sum((y_test - pred_100)**2) / np.sum((y_test - np.mean(y_test))**2))

print('--- Результати 100 епох ---')
print(f'MAE:  {mae_100:.4f} тис. $')
print(f'RMSE: {rmse_100:.4f} тис. $')
print(f'R²:   {r2_100:.4f}')

## 5. Експеримент 2: Навчання протягом 200 епох та порівняння

In [ ]:
model_200 = create_model()
history_200 = model_200.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

pred_200 = model_200.predict(X_test).flatten()
mae_200 = np.mean(np.abs(y_test - pred_200))
rmse_200 = np.sqrt(np.mean((y_test - pred_200)**2))
r2_200 = 1.0 - (np.sum((y_test - pred_200)**2) / np.sum((y_test - np.mean(y_test))**2))

print('--- Результати 200 епох ---')
print(f'MAE:  {mae_200:.4f} тис. $')
print(f'RMSE: {rmse_200:.4f} тис. $')
print(f'R²:   {r2_200:.4f}')

## 6. Візуалізація результатів

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_200.history['mae'], label='Навчальна помилка (MAE)')
plt.plot(history_200.history['val_mae'], label='Валідаційна помилка (Val MAE)')
plt.title('Динаміка зменшення помилки нейромережі протягом 200 епох')
plt.xlabel('Епоха')
plt.ylabel('MAE (тис. $)')
plt.legend()
plt.grid(True)
plt.show()